In [1]:
"""
Агрокалендарь (Алтайский край, обобщённые данные) с функциями прогноза и рекомендаций.
Один самодостаточный файл.
"""

import os
import requests
import pandas as pd
from dataclasses import dataclass, field
from datetime import date, timedelta
from typing import List, Optional, Dict, Any, Tuple


# ============================================================
# Вспомогательная функция – день года
# ============================================================
def doy(month: int, day: int) -> int:
    """Возвращает номер дня года (1 января = 1)."""
    return (date(2024, month, day) - date(2024, 1, 1)).days + 1


# ============================================================
# Базовые структуры данных
# ============================================================
@dataclass
class OperationRule:
    """Дополнительное погодное правило для операции."""
    param: str          # например 'precipitation_probability'
    condition: str      # 'lt', 'gt', 'le', 'ge', 'eq'
    value: float
    recommendation: str


@dataclass
class AgroOperation:
    """Агротехническая операция."""
    name: str
    window_start: int        # день года начала оптимального окна
    window_end: int          # день года окончания оптимального окна
    temp_min: Optional[float] = None
    temp_max: Optional[float] = None
    precip_max: Optional[float] = None   # мм/сут
    wind_max: Optional[float] = None     # м/с
    soil_moist_min: Optional[float] = None
    humidity_min: Optional[float] = None
    humidity_max: Optional[float] = None
    description: str = ""
    rules: List[OperationRule] = field(default_factory=list)


@dataclass
class GrowthStage:
    """Фенофаза (BBCH)."""
    name: str
    bbch_code: int
    operations: List[AgroOperation] = field(default_factory=list)


@dataclass
class Crop:
    """Сельскохозяйственная культура."""
    name_rus: str
    name_lat: str
    type_: str          # зерновая, зернобобовая, овощная, плодовая ...
    stages: List[GrowthStage] = field(default_factory=list)


# ============================================================
# Животноводство
# ============================================================
@dataclass
class VetProcedure:
    name: str
    window_start: int
    window_end: int
    temp_min: Optional[float] = None
    temp_max: Optional[float] = None
    precip_max: Optional[float] = None
    wind_max: Optional[float] = None
    soil_moist_min: Optional[float] = None      # добавлено для совместимости
    humidity_min: Optional[float] = None
    humidity_max: Optional[float] = None
    description: str = ""


@dataclass
class PhysiologicalState:
    name: str
    procedures: List[VetProcedure] = field(default_factory=list)


@dataclass
class Breed:
    name_rus: str
    states: List[PhysiologicalState] = field(default_factory=list)


@dataclass
class AnimalSpecies:
    name_rus: str
    breeds: List[Breed] = field(default_factory=list)


# ============================================================
# Блок предупреждений (болезни, вредители, стрессы)
# ============================================================
@dataclass
class PestDisease:
    name: str
    culture: str
    stage_sensitive: List[str] = field(default_factory=list)
    favorable_conditions: Dict[str, Any] = field(default_factory=dict)
    recommendation: str = ""


@dataclass
class ClimateStressor:
    name: str
    species: str
    threshold_params: Dict[str, float] = field(default_factory=dict)
    mitigation: str = ""


# ============================================================
# ЗЕРНОВЫЕ И ЗЕРНОБОБОВЫЕ
# ============================================================

wheat_winter = Crop(
    name_rus="Озимая пшеница",
    name_lat="Triticum aestivum L.",
    type_="зерновая",
    stages=[
        GrowthStage("Посев", 0, [
            AgroOperation("Посев озимой пшеницы", doy(8, 20), doy(9, 10),
                          temp_min=10, soil_moist_min=0.18,
                          description="Оптимальные сроки – третья декада августа – начало сентября."),
        ]),
        GrowthStage("Весеннее кущение", 25, [
            AgroOperation("Ранневесеннее боронование", doy(4, 15), doy(4, 30),
                          temp_min=5, description="Закрытие влаги."),
            AgroOperation("Азотная подкормка", doy(4, 20), doy(5, 10),
                          temp_min=5, precip_max=5, description="По мёрзло-талой почве."),
        ]),
        GrowthStage("Выход в трубку", 31, [
            AgroOperation("Гербицидная обработка", doy(5, 20), doy(6, 10),
                          temp_min=12, temp_max=22, precip_max=3, wind_max=5,
                          description="Фаза кущения – выхода в трубку."),
        ]),
        GrowthStage("Колошение", 55, [
            AgroOperation("Фунгицидная обработка", doy(6, 15), doy(7, 1),
                          temp_min=12, temp_max=25, precip_max=2, wind_max=5),
        ]),
        GrowthStage("Восковая спелость", 85, [
            AgroOperation("Уборка", doy(8, 1), doy(8, 20),
                          temp_min=15, precip_max=2,
                          description="Влажность зерна ≤ 14 %."),
        ]),
    ]
)

# --- Озимая рожь ---
winter_rye = Crop(
    name_rus="Озимая рожь",
    name_lat="Secale cereale L.",
    type_="зерновая",
    stages=[
        GrowthStage("Посев", 0, [
            AgroOperation("Посев озимой ржи", doy(8, 15), doy(9, 10),
                          temp_min=10, soil_moist_min=0.18,
                          description="Сроки близки к озимой пшенице; до 10 сентября."),
        ]),
        GrowthStage("Весеннее кущение", 25, [
            AgroOperation("Боронование", doy(4, 15), doy(4, 30), temp_min=5),
            AgroOperation("Азотная подкормка", doy(4, 20), doy(5, 5), temp_min=5),
        ]),
        GrowthStage("Выход в трубку", 31, [
            AgroOperation("Гербицидная обработка", doy(5, 15), doy(6, 5),
                          temp_min=12, temp_max=22, precip_max=3, wind_max=5),
        ]),
        GrowthStage("Колошение", 55, []),
        GrowthStage("Восковая спелость", 85, [
            AgroOperation("Уборка", doy(7, 25), doy(8, 15), temp_min=15, precip_max=2,
                          description="Убирают раньше пшеницы."),
        ]),
    ]
)

# --- Яровая пшеница ---
wheat_spring = Crop(
    name_rus="Яровая пшеница (мягкая)",
    name_lat="Triticum aestivum L.",
    type_="зерновая",
    stages=[
        GrowthStage("Посев", 0, [
            AgroOperation("Закрытие влаги", doy(4, 20), doy(5, 5), temp_min=5),
            AgroOperation("Посев", doy(5, 5), doy(5, 20), temp_min=8,
                          soil_moist_min=0.18, precip_max=5,
                          description="Прогрев почвы до +8…+10 °C."),
        ]),
        GrowthStage("Кущение", 25, [
            AgroOperation("Гербицидная обработка", doy(6, 1), doy(6, 15),
                          temp_min=12, temp_max=25, precip_max=3, wind_max=5),
        ]),
        GrowthStage("Выход в трубку", 31, [
            AgroOperation("Фунгицидная обработка", doy(6, 15), doy(7, 1),
                          temp_min=12, temp_max=25, precip_max=2, wind_max=5),
        ]),
        GrowthStage("Колошение", 55, [
            AgroOperation("Инсектицидная обработка", doy(7, 1), doy(7, 15),
                          temp_min=15, precip_max=1, wind_max=5),
        ]),
        GrowthStage("Восковая спелость", 85, [
            AgroOperation("Уборка", doy(8, 15), doy(9, 5), temp_min=15, precip_max=2),
        ]),
    ]
)

# --- Ячмень яровой ---
barley = Crop(
    name_rus="Ячмень яровой",
    name_lat="Hordeum vulgare L.",
    type_="зерновая",
    stages=[
        GrowthStage("Посев", 0, [
            AgroOperation("Посев", doy(5, 5), doy(5, 20), temp_min=7,
                          soil_moist_min=0.16, precip_max=5,
                          description="Ранний яровой сев."),
        ]),
        GrowthStage("Кущение", 25, [
            AgroOperation("Гербицидная обработка", doy(6, 1), doy(6, 15),
                          temp_min=12, temp_max=25, precip_max=3, wind_max=5),
        ]),
        GrowthStage("Колошение", 55, []),
        GrowthStage("Восковая спелость", 85, [
            AgroOperation("Уборка", doy(8, 1), doy(8, 20), temp_min=15, precip_max=2,
                          description="Вегетационный период 62–81 день."),
        ]),
    ]
)

# --- Овёс ---
oat = Crop(
    name_rus="Овёс посевной",
    name_lat="Avena sativa L.",
    type_="зерновая",
    stages=[
        GrowthStage("Посев", 0, [
            AgroOperation("Посев", doy(5, 1), doy(5, 15), temp_min=6,
                          soil_moist_min=0.16,
                          description="Ранний яровой сев."),
        ]),
        GrowthStage("Кущение", 25, []),
        GrowthStage("Вымётывание", 55, []),
        GrowthStage("Восковая спелость", 85, [
            AgroOperation("Уборка", doy(8, 10), doy(8, 30), temp_min=15, precip_max=2,
                          description="Вегетационный период 73–86 дней."),
        ]),
    ]
)

# --- Просо ---
millet = Crop(
    name_rus="Просо посевное",
    name_lat="Panicum miliaceum L.",
    type_="зерновая",
    stages=[
        GrowthStage("Посев", 0, [
            AgroOperation("Посев", doy(5, 20), doy(6, 5), temp_min=12,
                          precip_max=5, description="Теплолюбивая культура."),
        ]),
        GrowthStage("Вымётывание", 55, []),
        GrowthStage("Созревание", 85, [
            AgroOperation("Уборка", doy(8, 20), doy(9, 10), temp_min=15, precip_max=2,
                          description="Вегетационный период ~97 дней."),
        ]),
    ]
)

# --- Гречиха ---
buckwheat = Crop(
    name_rus="Гречиха посевная",
    name_lat="Fagopyrum esculentum Moench",
    type_="зерновая",
    stages=[
        GrowthStage("Посев", 0, [
            AgroOperation("Посев", doy(5, 25), doy(6, 10), temp_min=12,
                          soil_moist_min=0.16,
                          description="Поздняя культура; оптимально – первая декада июня."),
        ]),
        GrowthStage("Цветение", 65, []),
        GrowthStage("Созревание", 85, [
            AgroOperation("Уборка", doy(8, 25), doy(9, 15), temp_min=15, precip_max=2,
                          description="Вегетационный период 75–96 дней."),
        ]),
    ]
)

# --- Кукуруза ---
corn = Crop(
    name_rus="Кукуруза",
    name_lat="Zea mays L.",
    type_="зерновая",
    stages=[
        GrowthStage("Посев", 0, [
            AgroOperation("Посев", doy(5, 10), doy(5, 25), temp_min=10,
                          soil_moist_min=0.17, precip_max=5,
                          description="Прогрев почвы до +10…+12 °C."),
        ]),
        GrowthStage("4–6 листьев", 14, [
            AgroOperation("Междурядная культивация", doy(6, 5), doy(6, 20), precip_max=3),
        ]),
        GrowthStage("Цветение", 65, []),
        GrowthStage("Восковая спелость", 85, [
            AgroOperation("Уборка на зерно", doy(9, 1), doy(9, 30), temp_min=10, precip_max=2),
            AgroOperation("Уборка на силос", doy(8, 15), doy(9, 5), precip_max=5),
        ]),
    ]
)

# --- Горох ---
pea = Crop(
    name_rus="Горох посевной",
    name_lat="Pisum sativum L.",
    type_="зернобобовая",
    stages=[
        GrowthStage("Посев", 0, [
            AgroOperation("Посев", doy(4, 25), doy(5, 10), temp_min=5,
                          soil_moist_min=0.16, precip_max=5,
                          description="Холодостойкая культура; ранний сев."),
        ]),
        GrowthStage("Цветение", 65, []),
        GrowthStage("Созревание", 85, [
            AgroOperation("Уборка", doy(7, 20), doy(8, 10), temp_min=15, precip_max=2,
                          description="При пожелтении 45–60 % бобов."),
        ]),
    ]
)

# --- Нут ---
chickpea = Crop(
    name_rus="Нут",
    name_lat="Cicer arietinum L.",
    type_="зернобобовая",
    stages=[
        GrowthStage("Посев", 0, [
            AgroOperation("Посев", doy(4, 25), doy(5, 15), temp_min=5,
                          soil_moist_min=0.15,
                          description="Прогрев почвы до +5 °C; апрель–май."),
        ]),
        GrowthStage("Созревание", 85, [
            AgroOperation("Уборка", doy(8, 20), doy(9, 10), temp_min=15, precip_max=2,
                          description="Вегетационный период 90–102 дня."),
        ]),
    ]
)

# --- Соя ---
soy = Crop(
    name_rus="Соя",
    name_lat="Glycine max (L.) Merr.",
    type_="зернобобовая",
    stages=[
        GrowthStage("Посев", 0, [
            AgroOperation("Посев", doy(5, 20), doy(6, 5), temp_min=12,
                          soil_moist_min=0.17,
                          description="Оптимальный срок 25 мая – 5 июня."),
        ]),
        GrowthStage("Созревание", 85, [
            AgroOperation("Уборка", doy(9, 10), doy(9, 30), temp_min=10, precip_max=2,
                          description="Скороспелые сорта 83–99 дней."),
        ]),
    ]
)

# --- Чечевица ---
lentil = Crop(
    name_rus="Чечевица",
    name_lat="Lens culinaris Medik.",
    type_="зернобобовая",
    stages=[
        GrowthStage("Посев", 0, [
            AgroOperation("Посев", doy(4, 20), doy(5, 10), temp_min=5,
                          soil_moist_min=0.15,
                          description="Ранний сев, одновременно с ячменём."),
        ]),
        GrowthStage("Созревание", 85, [
            AgroOperation("Уборка", doy(8, 1), doy(8, 20), temp_min=15, precip_max=2,
                          description="Вегетационный период 75–110 дней."),
        ]),
    ]
)


# ============================================================
# МАСЛИЧНЫЕ И ТЕХНИЧЕСКИЕ
# ============================================================

sunflower = Crop(
    name_rus="Подсолнечник",
    name_lat="Helianthus annuus L.",
    type_="масличная",
    stages=[
        GrowthStage("Посев", 0, [
            AgroOperation("Посев", doy(5, 10), doy(5, 25), temp_min=10,
                          soil_moist_min=0.17, precip_max=5),
        ]),
        GrowthStage("4–6 листьев", 14, [
            AgroOperation("Междурядная культивация", doy(6, 5), doy(6, 20), precip_max=3),
        ]),
        GrowthStage("Цветение", 65, [
            AgroOperation("Обработка от вредителей", doy(7, 1), doy(7, 15),
                          temp_min=15, precip_max=1, wind_max=5),
        ]),
        GrowthStage("Созревание", 85, [
            AgroOperation("Уборка", doy(9, 15), doy(10, 10), precip_max=2,
                          description="Влажность семян 10–12 %."),
        ]),
    ]
)

# --- Лён масличный ---
flax = Crop(
    name_rus="Лён масличный",
    name_lat="Linum usitatissimum L.",
    type_="масличная",
    stages=[
        GrowthStage("Посев", 0, [
            AgroOperation("Посев", doy(5, 1), doy(5, 15), temp_min=6,
                          soil_moist_min=0.15,
                          description="Ранний яровой сев; заморозки до -5 °C переносит."),
        ]),
        GrowthStage("Цветение", 65, []),
        GrowthStage("Ранняя жёлтая спелость", 81, [
            AgroOperation("Уборка", doy(8, 10), doy(8, 30), temp_min=15, precip_max=2),
        ]),
    ]
)

# --- Рапс яровой ---
rapeseed = Crop(
    name_rus="Рапс яровой",
    name_lat="Brassica napus L.",
    type_="масличная",
    stages=[
        GrowthStage("Посев", 0, [
            AgroOperation("Посев", doy(5, 5), doy(5, 20), temp_min=8,
                          soil_moist_min=0.17, precip_max=5,
                          description="Ранний (до 10 мая) или поздний (25–31 мая) сев."),
        ]),
        GrowthStage("Стеблевание", 31, [
            AgroOperation("Гербицидная обработка", doy(5, 25), doy(6, 10),
                          temp_min=12, precip_max=3, wind_max=5),
        ]),
        GrowthStage("Созревание", 85, [
            AgroOperation("Уборка", doy(8, 20), doy(9, 10), temp_min=15, precip_max=2),
        ]),
    ]
)

# --- Горчица ---
mustard = Crop(
    name_rus="Горчица",
    name_lat="Brassica juncea (L.) Czern.",
    type_="масличная",
    stages=[
        GrowthStage("Посев", 0, [
            AgroOperation("Посев", doy(4, 25), doy(5, 15), temp_min=6,
                          description="Ранневесенний сев; всходы переносят заморозки."),
        ]),
        GrowthStage("Созревание", 85, [
            AgroOperation("Уборка", doy(8, 10), doy(8, 30), temp_min=15, precip_max=2,
                          description="Вегетационный период 105–110 дней."),
        ]),
    ]
)

# --- Рыжик ---
camelina = Crop(
    name_rus="Рыжик масличный",
    name_lat="Camelina sativa (L.) Crantz",
    type_="масличная",
    stages=[
        GrowthStage("Посев", 0, [
            AgroOperation("Посев", doy(4, 25), doy(5, 10), temp_min=1,
                          soil_moist_min=0.14,
                          description="Самый ранний сев; заморозки до -12 °C."),
        ]),
        GrowthStage("Созревание", 85, [
            AgroOperation("Уборка", doy(7, 20), doy(8, 10), temp_min=15, precip_max=2,
                          description="Вегетационный период 66–90 дней."),
        ]),
    ]
)

# ============================================================
# КОРМОВЫЕ ТРАВЫ
# ============================================================

alfalfa = Crop(
    name_rus="Люцерна синегибридная",
    name_lat="Medicago varia Martyn",
    type_="кормовая",
    stages=[
        GrowthStage("Посев", 0, [
            AgroOperation("Посев (беспокровный)", doy(4, 25), doy(5, 15), temp_min=10,
                          soil_moist_min=0.17,
                          description="Ранневесенний или летний (июнь) посев."),
        ]),
        GrowthStage("Бутонизация", 55, [
            AgroOperation("Первый укос (сено/сенаж)", doy(6, 15), doy(7, 5), precip_max=1),
        ]),
        GrowthStage("Отрастание", 0, [
            AgroOperation("Второй укос", doy(7, 25), doy(8, 15), precip_max=2),
        ]),
        GrowthStage("Цветение (семенники)", 65, [
            AgroOperation("Уборка семян", doy(8, 15), doy(8, 30), precip_max=1),
        ]),
    ]
)

sainfoin = Crop(
    name_rus="Эспарцет песчаный",
    name_lat="Onobrychis arenaria (Kit.) DC.",
    type_="кормовая",
    stages=[
        GrowthStage("Посев", 0, [
            AgroOperation("Посев", doy(4, 20), doy(6, 15), temp_min=6,
                          soil_moist_min=0.15,
                          description="Ранневесенний или летний посев."),
        ]),
        GrowthStage("Бутонизация", 55, [
            AgroOperation("Первый укос", doy(6, 15), doy(7, 1), precip_max=1,
                          description="До укосной спелости 60–70 дней."),
        ]),
    ]
)

clover = Crop(
    name_rus="Клевер луговой",
    name_lat="Trifolium pratense L.",
    type_="кормовая",
    stages=[
        GrowthStage("Посев", 0, [
            AgroOperation("Посев", doy(4, 25), doy(5, 15), temp_min=10,
                          description="Ранневесенний подпокровный или беспокровный."),
        ]),
        GrowthStage("Бутонизация", 55, [
            AgroOperation("Первый укос", doy(6, 20), doy(7, 5), precip_max=1),
        ]),
        GrowthStage("Отрастание", 0, [
            AgroOperation("Второй укос", doy(8, 1), doy(8, 20), precip_max=2),
        ]),
    ]
)

brome = Crop(
    name_rus="Кострец безостый",
    name_lat="Bromopsis inermis (Leyss.) Holub",
    type_="кормовая",
    stages=[
        GrowthStage("Посев", 0, [
            AgroOperation("Посев", doy(4, 20), doy(5, 15), temp_min=8,
                          description="Весенний или осенний посев."),
        ]),
        GrowthStage("Выметывание", 55, [
            AgroOperation("Укос", doy(6, 20), doy(7, 10), precip_max=2),
        ]),
    ]
)

wheatgrass = Crop(
    name_rus="Житняк",
    name_lat="Agropyron cristatum (L.) Gaertn.",
    type_="кормовая",
    stages=[
        GrowthStage("Посев", 0, [
            AgroOperation("Посев", doy(4, 20), doy(5, 10), temp_min=6,
                          soil_moist_min=0.14,
                          description="Ранневесенний или осенний сев."),
        ]),
        GrowthStage("Колошение", 55, [
            AgroOperation("Укос", doy(6, 15), doy(7, 1), precip_max=2),
        ]),
    ]
)

vetch = Crop(
    name_rus="Вика яровая",
    name_lat="Vicia sativa L.",
    type_="кормовая",
    stages=[
        GrowthStage("Посев", 0, [
            AgroOperation("Посев", doy(4, 25), doy(5, 10), temp_min=5,
                          description="Холодостойкая; ранний сев."),
        ]),
        GrowthStage("Цветение", 65, [
            AgroOperation("Укос на зелёный корм", doy(6, 20), doy(7, 10), precip_max=3),
        ]),
        GrowthStage("Созревание", 85, [
            AgroOperation("Уборка на семена", doy(8, 1), doy(8, 20), precip_max=2),
        ]),
    ]
)

sudan_grass = Crop(
    name_rus="Суданская трава",
    name_lat="Sorghum sudanense (Piper) Stapf",
    type_="кормовая",
    stages=[
        GrowthStage("Посев", 0, [
            AgroOperation("Посев", doy(5, 20), doy(6, 15), temp_min=12,
                          description="Третья декада мая – середина июня."),
        ]),
        GrowthStage("Выметывание", 55, [
            AgroOperation("Первый укос", doy(7, 10), doy(7, 25), precip_max=3,
                          description="46–48 дней от всходов до вымётывания."),
        ]),
        GrowthStage("Отрастание", 0, [
            AgroOperation("Второй укос", doy(8, 10), doy(8, 30), precip_max=3),
        ]),
    ]
)

mohar = Crop(
    name_rus="Могар",
    name_lat="Setaria italica subsp. moharia",
    type_="кормовая",
    stages=[
        GrowthStage("Посев", 0, [
            AgroOperation("Посев", doy(5, 20), doy(6, 10), temp_min=12,
                          description="Одновременно с просом и суданской травой."),
        ]),
        GrowthStage("Выметывание", 55, [
            AgroOperation("Укос на сено", doy(7, 15), doy(7, 30), precip_max=3),
        ]),
    ]
)

ryegrass = Crop(
    name_rus="Райграс однолетний",
    name_lat="Lolium multiflorum Lam.",
    type_="кормовая",
    stages=[
        GrowthStage("Посев", 0, [
            AgroOperation("Посев", doy(4, 25), doy(5, 15), temp_min=8,
                          description="Конец апреля – начало мая."),
        ]),
        GrowthStage("Колошение", 55, [
            AgroOperation("Укос", doy(6, 15), doy(7, 1), precip_max=3),
        ]),
        GrowthStage("Отрастание", 0, [
            AgroOperation("Второй укос", doy(7, 20), doy(8, 10), precip_max=3),
        ]),
    ]
)

# ============================================================
# ОВОЩНЫЕ КУЛЬТУРЫ
# ============================================================

potato = Crop(
    name_rus="Картофель продовольственный",
    name_lat="Solanum tuberosum L.",
    type_="овощная",
    stages=[
        GrowthStage("Посадка", 0, [
            AgroOperation("Посадка", doy(5, 15), doy(6, 5), temp_min=7,
                          soil_moist_min=0.16,
                          description="Прогрев почвы на 10 см до +7…+8 °C."),
        ]),
        GrowthStage("Появление всходов", 11, [
            AgroOperation("Рыхление междурядий", doy(6, 1), doy(6, 15), precip_max=5),
        ]),
        GrowthStage("Бутонизация", 51, [
            AgroOperation("Окучивание и подкормка", doy(6, 20), doy(7, 10), precip_max=5),
            AgroOperation("Профилактика фитофтороза", doy(6, 25), doy(7, 15),
                          temp_min=10, temp_max=25, humidity_min=70, precip_max=2,
                          description="При влажности >75 % – риск фитофторы."),
        ]),
        GrowthStage("Уборка", 91, [
            AgroOperation("Уборка ранних сортов", doy(8, 1), doy(8, 20), precip_max=3),
            AgroOperation("Уборка поздних сортов", doy(8, 25), doy(9, 15), temp_min=10, precip_max=3),
        ]),
    ]
)

cabbage = Crop(
    name_rus="Капуста белокочанная",
    name_lat="Brassica oleracea var. capitata",
    type_="овощная",
    stages=[
        GrowthStage("Высадка рассады", 0, [
            AgroOperation("Высадка в грунт", doy(5, 15), doy(6, 1), temp_min=10, temp_max=25,
                          precip_max=10, description="После угрозы заморозков."),
        ]),
        GrowthStage("Завязывание кочана", 41, [
            AgroOperation("Подкормка и полив", doy(6, 15), doy(7, 15), temp_max=28),
        ]),
        GrowthStage("Техническая спелость", 49, [
            AgroOperation("Уборка ранней", doy(7, 1), doy(7, 20), precip_max=5),
            AgroOperation("Уборка средней", doy(8, 1), doy(8, 20), precip_max=5),
            AgroOperation("Уборка поздней", doy(9, 1), doy(9, 30), precip_max=5),
        ]),
    ]
)

carrot = Crop(
    name_rus="Морковь столовая",
    name_lat="Daucus carota L.",
    type_="овощная",
    stages=[
        GrowthStage("Посев", 0, [
            AgroOperation("Посев", doy(4, 25), doy(5, 15), temp_min=8,
                          soil_moist_min=0.12,
                          description="Ранневесенний посев; всходы при +8 °C."),
        ]),
        GrowthStage("Прореживание", 12, [
            AgroOperation("Прореживание", doy(5, 25), doy(6, 15)),
        ]),
        GrowthStage("Уборка", 49, [
            AgroOperation("Уборка", doy(8, 20), doy(9, 20), precip_max=3),
        ]),
    ]
)

beet = Crop(
    name_rus="Свёкла столовая",
    name_lat="Beta vulgaris L.",
    type_="овощная",
    stages=[
        GrowthStage("Посев", 0, [
            AgroOperation("Посев", doy(4, 25), doy(5, 15), temp_min=10,
                          description="Конец апреля – начало мая."),
        ]),
        GrowthStage("Прореживание", 12, [
            AgroOperation("Прореживание", doy(5, 25), doy(6, 10)),
        ]),
        GrowthStage("Уборка", 49, [
            AgroOperation("Уборка", doy(8, 20), doy(9, 20), precip_max=3),
        ]),
    ]
)

onion = Crop(
    name_rus="Лук репчатый",
    name_lat="Allium cepa L.",
    type_="овощная",
    stages=[
        GrowthStage("Посадка севка", 0, [
            AgroOperation("Посадка севка", doy(5, 1), doy(5, 15), temp_min=8,
                          description="С 1 по 15 мая."),
        ]),
        GrowthStage("Формирование луковиц", 43, [
            AgroOperation("Подкормка, рыхление", doy(6, 1), doy(7, 1)),
        ]),
        GrowthStage("Полегание пера", 49, [
            AgroOperation("Уборка", doy(8, 1), doy(8, 20), precip_max=2),
        ]),
    ]
)

tomato = Crop(
    name_rus="Томат",
    name_lat="Solanum lycopersicum L.",
    type_="овощная",
    stages=[
        GrowthStage("Высадка рассады", 0, [
            AgroOperation("Высадка в открытый грунт", doy(6, 1), doy(6, 15), temp_min=12,
                          description="После угрозы заморозков (начало июня)."),
        ]),
        GrowthStage("Цветение", 65, [
            AgroOperation("Подкормка, пасынкование", doy(6, 25), doy(7, 15)),
        ]),
        GrowthStage("Плодоношение", 71, [
            AgroOperation("Сбор урожая", doy(7, 20), doy(9, 10)),
        ]),
    ]
)

cucumber = Crop(
    name_rus="Огурец",
    name_lat="Cucumis sativus L.",
    type_="овощная",
    stages=[
        GrowthStage("Посев / высадка", 0, [
            AgroOperation("Посев в открытый грунт", doy(5, 25), doy(6, 5), temp_min=15,
                          soil_moist_min=0.17,
                          description="Прогрев почвы до +15…+18 °C; конец мая."),
        ]),
        GrowthStage("Цветение", 65, []),
        GrowthStage("Плодоношение", 71, [
            AgroOperation("Сбор урожая", doy(7, 1), doy(8, 31)),
        ]),
    ]
)

pepper = Crop(
    name_rus="Перец сладкий",
    name_lat="Capsicum annuum L.",
    type_="овощная",
    stages=[
        GrowthStage("Высадка рассады", 0, [
            AgroOperation("Высадка в открытый грунт", doy(6, 5), doy(6, 20), temp_min=15,
                          description="После угрозы заморозков."),
        ]),
        GrowthStage("Плодоношение", 71, [
            AgroOperation("Сбор урожая", doy(7, 20), doy(8, 31)),
        ]),
    ]
)

eggplant = Crop(
    name_rus="Баклажан",
    name_lat="Solanum melongena L.",
    type_="овощная",
    stages=[
        GrowthStage("Высадка рассады", 0, [
            AgroOperation("Высадка в открытый грунт", doy(6, 5), doy(6, 20), temp_min=15,
                          description="После угрозы заморозков."),
        ]),
        GrowthStage("Плодоношение", 71, [
            AgroOperation("Сбор урожая", doy(7, 25), doy(8, 31)),
        ]),
    ]
)

watermelon = Crop(
    name_rus="Арбуз",
    name_lat="Citrullus lanatus (Thunb.) Matsum. & Nakai",
    type_="бахчевая",
    stages=[
        GrowthStage("Посев / высадка", 0, [
            AgroOperation("Посев / высадка рассады", doy(5, 25), doy(6, 5), temp_min=15,
                          description="Прогрев почвы до +15…+18 °C."),
        ]),
        GrowthStage("Созревание", 85, [
            AgroOperation("Сбор урожая", doy(8, 10), doy(9, 10)),
        ]),
    ]
)

melon = Crop(
    name_rus="Дыня",
    name_lat="Cucumis melo L.",
    type_="бахчевая",
    stages=[
        GrowthStage("Посев / высадка", 0, [
            AgroOperation("Высадка рассады", doy(5, 25), doy(6, 10), temp_min=15,
                          description="После угрозы заморозков."),
        ]),
        GrowthStage("Созревание", 85, [
            AgroOperation("Сбор урожая", doy(8, 10), doy(9, 5)),
        ]),
    ]
)

sugar_beet = Crop(
    name_rus="Сахарная свёкла",
    name_lat="Beta vulgaris L. saccharifera",
    type_="техническая",
    stages=[
        GrowthStage("Посев", 0, [
            AgroOperation("Посев", doy(4, 25), doy(5, 15), temp_min=8,
                          soil_moist_min=0.16,
                          description="Ранневесенний сев."),
        ]),
        GrowthStage("Смыкание рядков", 33, []),
        GrowthStage("Уборка", 49, [
            AgroOperation("Копка", doy(9, 1), doy(10, 15), temp_min=5, precip_max=5,
                          description="Массовая копка с середины сентября."),
        ]),
    ]
)

# ============================================================
# ПЛОДОВО-ЯГОДНЫЕ
# ============================================================

apple = Crop(
    name_rus="Яблоня (полукультурка)",
    name_lat="Malus domestica",
    type_="плодовая",
    stages=[
        GrowthStage("Цветение", 69, [
            AgroOperation("Обработка от парши и плодожорки", doy(5, 5), doy(5, 20),
                          temp_min=12, precip_max=1, wind_max=5,
                          description="До и после цветения."),
        ]),
        GrowthStage("Рост плодов", 75, [
            AgroOperation("Подкормка и орошение", doy(6, 1), doy(7, 30), temp_max=30),
        ]),
        GrowthStage("Съёмная зрелость", 87, [
            AgroOperation("Сбор урожая", doy(8, 20), doy(9, 20), precip_max=3,
                          description="Для летних сортов – август, осенних – сентябрь."),
        ]),
    ]
)

cherry = Crop(
    name_rus="Вишня обыкновенная и войлочная",
    name_lat="Prunus cerasus L. / Prunus tomentosa Thunb.",
    type_="плодовая",
    stages=[
        GrowthStage("Цветение", 69, [
            AgroOperation("Обработка от монилиоза", doy(5, 5), doy(5, 15),
                          temp_min=12, precip_max=1, wind_max=5),
        ]),
        GrowthStage("Созревание", 87, [
            AgroOperation("Сбор урожая", doy(7, 1), doy(8, 5), precip_max=3,
                          description="Ранние – конец июня, поздние – начало августа."),
        ]),
    ]
)

plum = Crop(
    name_rus="Слива",
    name_lat="Prunus domestica L.",
    type_="плодовая",
    stages=[
        GrowthStage("Цветение", 69, [
            AgroOperation("Обработка от вредителей", doy(5, 10), doy(5, 20),
                          temp_min=12, precip_max=1, wind_max=5),
        ]),
        GrowthStage("Созревание", 87, [
            AgroOperation("Сбор урожая", doy(8, 1), doy(8, 31), precip_max=3,
                          description="Ранние – первая декада августа."),
        ]),
    ]
)

pear = Crop(
    name_rus="Груша уссурийская",
    name_lat="Pyrus ussuriensis Maxim.",
    type_="плодовая",
    stages=[
        GrowthStage("Цветение", 69, [
            AgroOperation("Обработка от грушевой плодожорки", doy(5, 15), doy(5, 25),
                          temp_min=12, precip_max=1, wind_max=5),
        ]),
        GrowthStage("Созревание", 87, [
            AgroOperation("Сбор урожая", doy(8, 20), doy(9, 30), precip_max=3,
                          description="Летние – конец августа, осенние – сентябрь."),
        ]),
    ]
)

currant_black = Crop(
    name_rus="Смородина чёрная",
    name_lat="Ribes nigrum L.",
    type_="ягодная",
    stages=[
        GrowthStage("Распускание почек", 11, [
            AgroOperation("Обработка от почкового клеща", doy(4, 15), doy(4, 30),
                          temp_min=5),
        ]),
        GrowthStage("Цветение", 69, [
            AgroOperation("Подкормка", doy(5, 15), doy(5, 30)),
        ]),
        GrowthStage("Созревание", 87, [
            AgroOperation("Сбор урожая", doy(7, 10), doy(8, 5),
                          description="Созревание 16–22 июля."),
        ]),
    ]
)

raspberry = Crop(
    name_rus="Малина",
    name_lat="Rubus idaeus L.",
    type_="ягодная",
    stages=[
        GrowthStage("Отрастание побегов", 31, [
            AgroOperation("Подкормка, полив", doy(5, 1), doy(5, 31)),
        ]),
        GrowthStage("Цветение", 69, [
            AgroOperation("Обработка от малинного жука", doy(6, 1), doy(6, 15),
                          temp_min=15, precip_max=1, wind_max=5),
        ]),
        GrowthStage("Созревание", 87, [
            AgroOperation("Сбор урожая", doy(7, 10), doy(8, 20),
                          description="Основной сбор – вторая половина июля."),
        ]),
    ]
)

honeysuckle = Crop(
    name_rus="Жимолость синяя",
    name_lat="Lonicera caerulea L.",
    type_="ягодная",
    stages=[
        GrowthStage("Цветение", 69, [
            AgroOperation("Подкормка", doy(4, 20), doy(5, 5)),
        ]),
        GrowthStage("Созревание", 87, [
            AgroOperation("Сбор урожая", doy(6, 15), doy(7, 5),
                          description="Самая ранняя ягода."),
        ]),
    ]
)

sea_buckthorn = Crop(
    name_rus="Облепиха крушиновидная",
    name_lat="Hippophae rhamnoides L.",
    type_="ягодная",
    stages=[
        GrowthStage("Цветение", 69, [
            AgroOperation("Подкормка", doy(5, 1), doy(5, 15)),
        ]),
        GrowthStage("Созревание", 87, [
            AgroOperation("Сбор урожая", doy(8, 15), doy(9, 15),
                          description="Техническая спелость – август–сентябрь."),
        ]),
    ]
)

strawberry = Crop(
    name_rus="Клубника садовая",
    name_lat="Fragaria × ananassa Duchesne",
    type_="ягодная",
    stages=[
        GrowthStage("Начало вегетации", 11, [
            AgroOperation("Очистка, подкормка", doy(4, 15), doy(5, 1), temp_min=5),
        ]),
        GrowthStage("Цветение", 69, [
            AgroOperation("Обработка от серой гнили", doy(5, 20), doy(6, 5),
                          temp_min=12, precip_max=1, wind_max=5),
        ]),
        GrowthStage("Плодоношение", 87, [
            AgroOperation("Сбор урожая", doy(6, 20), doy(7, 20), precip_max=5),
        ]),
    ]
)

# ============================================================
# ЖИВОТНОВОДСТВО
# ============================================================

cattle = AnimalSpecies(
    name_rus="КРС (молочный)",
    breeds=[
        Breed("Чёрно-пёстрая", [
            PhysiologicalState("Сухостойный период", [
                VetProcedure("Запуск коров", doy(1, 15), doy(2, 15), temp_min=-10, temp_max=15,
                             description="Постепенное снижение доения."),
            ]),
            PhysiologicalState("Отёл", [
                VetProcedure("Подготовка к отёлу", doy(2, 20), doy(4, 10), temp_min=0, temp_max=20,
                             description="Усиленное кормление, контроль температуры."),
                VetProcedure("Вакцинация телят", doy(3, 1), doy(4, 30), temp_min=5, temp_max=25),
            ]),
            PhysiologicalState("Лактация", [
                VetProcedure("Выгон на пастбище", doy(5, 1), doy(5, 15), temp_min=10,
                             description="Устойчивый переход t > +10 °C."),
                VetProcedure("Заготовка сена", doy(6, 20), doy(7, 10), precip_max=1),
            ]),
            PhysiologicalState("Осеменение", [
                VetProcedure("Случка / искусственное осеменение", doy(9, 1), doy(9, 20),
                             temp_min=10, temp_max=25),
            ]),
            PhysiologicalState("Стойловый период", [
                VetProcedure("Переход на стойловое содержание", doy(10, 1), doy(10, 20),
                             temp_min=5, temp_max=15),
            ]),
        ]),
        Breed("Симментальская", [
            PhysiologicalState("Отёл", [
                VetProcedure("Подготовка к отёлу", doy(2, 20), doy(4, 10), temp_min=0, temp_max=20),
            ]),
            PhysiologicalState("Лактация", [
                VetProcedure("Выгон на пастбище", doy(5, 1), doy(5, 15), temp_min=10),
            ]),
            PhysiologicalState("Осеменение", [
                VetProcedure("Случка", doy(9, 1), doy(9, 20), temp_min=10, temp_max=25),
            ]),
        ]),
        Breed("Герефордская", [
            PhysiologicalState("Отёл", [
                VetProcedure("Подготовка к отёлу", doy(3, 1), doy(4, 15), temp_min=0, temp_max=20),
            ]),
            PhysiologicalState("Лактация", [
                VetProcedure("Выгон на пастбище", doy(5, 1), doy(5, 15), temp_min=10),
            ]),
        ]),
    ]
)

sheep = AnimalSpecies(
    name_rus="Овцы",
    breeds=[
        Breed("Алтайская порода", [
            PhysiologicalState("Окот", [
                VetProcedure("Окот", doy(3, 15), doy(4, 10), temp_min=0, temp_max=15,
                             description="Массовый окот в апреле."),
            ]),
            PhysiologicalState("Пастбищный период", [
                VetProcedure("Стрижка", doy(5, 20), doy(6, 10), temp_min=15, precip_max=1),
                VetProcedure("Купка (противопаразитарная)", doy(6, 1), doy(6, 20), temp_min=15, precip_max=2),
            ]),
            PhysiologicalState("Случка", [
                VetProcedure("Случка овец", doy(8, 15), doy(9, 15), temp_min=10, temp_max=25),
            ]),
        ]),
        Breed("Эдильбаевская", [
            PhysiologicalState("Окот", [
                VetProcedure("Окот", doy(3, 15), doy(4, 10), temp_min=0, temp_max=15),
            ]),
            PhysiologicalState("Пастбищный период", [
                VetProcedure("Стрижка", doy(5, 25), doy(6, 10), temp_min=15),
            ]),
        ]),
    ]
)

pigs = AnimalSpecies(
    name_rus="Свиньи",
    breeds=[
        Breed("Крупная белая", [
            PhysiologicalState("Супоросность", [
                VetProcedure("Вакцинация супоросных маток", doy(2, 1), doy(3, 31),
                             temp_min=10, temp_max=25),
            ]),
            PhysiologicalState("Опорос", [
                VetProcedure("Подготовка станка, обогрев", doy(3, 1), doy(4, 15),
                             temp_min=20, temp_max=28,
                             description="Оптимальная температура 20–22 °C."),
            ]),
            PhysiologicalState("Доращивание", [
                VetProcedure("Выгул молодняка", doy(4, 20), doy(9, 30),
                             temp_min=12, temp_max=30, precip_max=5),
                VetProcedure("Обработка от гельминтов", doy(5, 1), doy(5, 15),
                             temp_min=15, temp_max=25),
            ]),
        ]),
        Breed("Ландрас", [
            PhysiologicalState("Опорос", [
                VetProcedure("Подготовка станка", doy(3, 1), doy(4, 15),
                             temp_min=20, temp_max=28),
            ]),
        ]),
        Breed("Дюрок", [
            PhysiologicalState("Опорос", [
                VetProcedure("Подготовка станка", doy(3, 1), doy(4, 15),
                             temp_min=20, temp_max=28),
            ]),
        ]),
    ]
)

goats = AnimalSpecies(
    name_rus="Козы",
    breeds=[
        Breed("Алтайская пуховая", [
            PhysiologicalState("Козление", [
                VetProcedure("Козление", doy(3, 20), doy(4, 20), temp_min=0, temp_max=15),
            ]),
            PhysiologicalState("Пастбищный период", [
                VetProcedure("Ческа пуха", doy(5, 1), doy(5, 31), temp_min=15),
                VetProcedure("Выгон на пастбище", doy(5, 1), doy(5, 15), temp_min=10),
            ]),
            PhysiologicalState("Случка", [
                VetProcedure("Случка", doy(11, 1), doy(12, 10), temp_min=-5, temp_max=5,
                             description="Брачный сезон – ноябрь–декабрь."),
            ]),
        ]),
    ]
)

horses = AnimalSpecies(
    name_rus="Лошади",
    breeds=[
        Breed("Алтайская порода", [
            PhysiologicalState("Выжеребка", [
                VetProcedure("Подготовка к выжеребке", doy(3, 1), doy(4, 30),
                             temp_min=0, temp_max=15,
                             description="Выжеребка через ~11 мес. после случки."),
            ]),
            PhysiologicalState("Случка", [
                VetProcedure("Случка (табунная)", doy(4, 15), doy(8, 1),
                             temp_min=10, temp_max=30),
            ]),
            PhysiologicalState("Пастбищный период", [
                VetProcedure("Выгон на пастбище", doy(5, 1), doy(5, 15), temp_min=10),
            ]),
        ]),
        Breed("Орловский рысак", [
            PhysiologicalState("Выжеребка", [
                VetProcedure("Подготовка к выжеребке", doy(3, 1), doy(4, 30),
                             temp_min=0, temp_max=15),
            ]),
            PhysiologicalState("Случка", [
                VetProcedure("Случка (конюшенная)", doy(2, 15), doy(7, 15),
                             temp_min=5, temp_max=25),
            ]),
        ]),
    ]
)

poultry = AnimalSpecies(
    name_rus="Птица",
    breeds=[
        Breed("Куры-несушки", [
            PhysiologicalState("Яйцекладка", [
                VetProcedure("Обеспечение светового режима", doy(1, 1), doy(12, 31),
                             temp_min=10, temp_max=28,
                             description="16 ч света ежедневно."),
                VetProcedure("Вакцинация (Ньюкасл)", doy(3, 1), doy(3, 15),
                             temp_min=15, temp_max=25),
                VetProcedure("Профилактика кокцидиоза", doy(4, 15), doy(5, 1),
                             temp_min=15, temp_max=25),
            ]),
        ]),
        Breed("Бройлеры", [
            PhysiologicalState("Выращивание", [
                VetProcedure("Вакцинация (Марек, Гамборо)", doy(1, 1), doy(12, 31),
                             temp_min=18, temp_max=28),
                VetProcedure("Убой (42–45 дней)", doy(1, 1), doy(12, 31)),
            ]),
        ]),
        Breed("Гуси", [
            PhysiologicalState("Продуктивный период", [
                VetProcedure("Яйцекладка (февраль–июнь)", doy(2, 1), doy(6, 30),
                             temp_min=0, temp_max=25),
                VetProcedure("Убой молодняка (август–сентябрь)", doy(8, 1), doy(9, 30)),
            ]),
        ]),
        Breed("Утки", [
            PhysiologicalState("Продуктивный период", [
                VetProcedure("Яйцекладка (апрель–август)", doy(4, 1), doy(8, 31),
                             temp_min=10, temp_max=28),
                VetProcedure("Убой молодняка (август–октябрь)", doy(8, 1), doy(10, 31)),
            ]),
        ]),
        Breed("Индейки", [
            PhysiologicalState("Продуктивный период", [
                VetProcedure("Яйцекладка (апрель–август)", doy(4, 1), doy(8, 31),
                             temp_min=10, temp_max=25),
                VetProcedure("Убой молодняка (октябрь–декабрь)", doy(10, 1), doy(12, 31)),
            ]),
        ]),
    ]
)

rabbits = AnimalSpecies(
    name_rus="Кролики",
    breeds=[
        Breed("Калифорнийская", [
            PhysiologicalState("Окрол", [
                VetProcedure("Подготовка клеток, обогрев", doy(2, 1), doy(4, 30),
                             temp_min=5, temp_max=25,
                             description="Первый окрол – конец февраля."),
            ]),
            PhysiologicalState("Откорм", [
                VetProcedure("Отсадка и откорм", doy(3, 1), doy(10, 31),
                             description="Интенсивный откорм 30–45 дней."),
            ]),
            PhysiologicalState("Профилактика", [
                VetProcedure("Вакцинация (миксоматоз, ВГБК)", doy(3, 1), doy(4, 30),
                             temp_min=10, temp_max=25),
            ]),
        ]),
        Breed("Белая новозеландская", [
            PhysiologicalState("Окрол", [
                VetProcedure("Подготовка клеток", doy(2, 1), doy(4, 30), temp_min=5, temp_max=25),
            ]),
            PhysiologicalState("Откорм", [
                VetProcedure("Отсадка и откорм", doy(3, 1), doy(10, 31)),
            ]),
        ]),
    ]
)

fur_animals = AnimalSpecies(
    name_rus="Пушные звери",
    breeds=[
        Breed("Норка", [
            PhysiologicalState("Подготовка к гону", [
                VetProcedure("Контроль упитанности, витаминизация", doy(2, 1), doy(2, 28)),
            ]),
            PhysiologicalState("Гон", [
                VetProcedure("Гон", doy(3, 1), doy(3, 20), temp_min=-5, temp_max=10),
            ]),
            PhysiologicalState("Щенение", [
                VetProcedure("Подготовка домиков, контроль", doy(4, 25), doy(5, 15),
                             temp_min=10, temp_max=25),
            ]),
            PhysiologicalState("Выращивание молодняка", [
                VetProcedure("Отсадка, кормление", doy(6, 1), doy(9, 30)),
            ]),
            PhysiologicalState("Убой", [
                VetProcedure("Забой на мех", doy(10, 1), doy(11, 30), temp_max=5),
            ]),
        ]),
        Breed("Соболь", [
            PhysiologicalState("Гон", [
                VetProcedure("Гон", doy(7, 1), doy(7, 31), temp_min=10, temp_max=25),
            ]),
            PhysiologicalState("Щенение", [
                VetProcedure("Подготовка домиков", doy(3, 20), doy(4, 15), temp_min=0, temp_max=15),
            ]),
            PhysiologicalState("Убой", [
                VetProcedure("Забой на мех", doy(10, 1), doy(11, 15), temp_max=5),
            ]),
        ]),
        Breed("Лисица", [
            PhysiologicalState("Гон", [
                VetProcedure("Гон", doy(1, 25), doy(2, 20), temp_min=-10, temp_max=5),
            ]),
            PhysiologicalState("Щенение", [
                VetProcedure("Подготовка домиков", doy(3, 25), doy(4, 20), temp_min=0, temp_max=15),
            ]),
            PhysiologicalState("Убой", [
                VetProcedure("Забой на мех", doy(10, 1), doy(11, 30), temp_max=5),
            ]),
        ]),
    ]
)

bees = AnimalSpecies(
    name_rus="Пчёлы медоносные",
    breeds=[
        Breed("Среднерусская", [
            PhysiologicalState("Весеннее развитие", [
                VetProcedure("Выставка ульев из зимовника", doy(3, 25), doy(4, 15),
                             temp_min=8, precip_max=2,
                             description="Первый облёт при t > 8 °C."),
            ]),
            PhysiologicalState("Медосбор", [
                VetProcedure("Кочевка на медоносы", doy(6, 15), doy(7, 15),
                             temp_min=15, precip_max=3, wind_max=8),
            ]),
            PhysiologicalState("Осеннее размножение", [
                VetProcedure("Сборка гнёзд на зиму", doy(8, 25), doy(9, 10), temp_min=10),
            ]),
            PhysiologicalState("Зимовка", [
                VetProcedure("Установка ульев в зимовник", doy(10, 25), doy(11, 10),
                             temp_min=-5, temp_max=5),
            ]),
        ]),
    ]
)

# ============================================================
# СВОДНЫЕ СЛОВАРИ
# ============================================================
ALL_CROPS = {
    "озимая пшеница": wheat_winter,
    "озимая рожь": winter_rye,
    "яровая пшеница": wheat_spring,
    "ячмень": barley,
    "овёс": oat,
    "просо": millet,
    "гречиха": buckwheat,
    "кукуруза": corn,
    "горох": pea,
    "нут": chickpea,
    "соя": soy,
    "чечевица": lentil,
    "подсолнечник": sunflower,
    "лён масличный": flax,
    "рапс": rapeseed,
    "горчица": mustard,
    "рыжик": camelina,
    "люцерна": alfalfa,
    "эспарцет": sainfoin,
    "клевер": clover,
    "кострец": brome,
    "житняк": wheatgrass,
    "вика": vetch,
    "суданская трава": sudan_grass,
    "могар": mohar,
    "райграс": ryegrass,
    "картофель": potato,
    "капуста": cabbage,
    "морковь": carrot,
    "свёкла столовая": beet,
    "лук": onion,
    "томат": tomato,
    "огурец": cucumber,
    "перец": pepper,
    "баклажан": eggplant,
    "арбуз": watermelon,
    "дыня": melon,
    "сахарная свёкла": sugar_beet,
    "яблоня": apple,
    "вишня": cherry,
    "слива": plum,
    "груша": pear,
    "смородина чёрная": currant_black,
    "малина": raspberry,
    "жимолость": honeysuckle,
    "облепиха": sea_buckthorn,
    "клубника": strawberry,
}

ALL_ANIMALS = {
    "КРС": cattle,
    "овцы": sheep,
    "свиньи": pigs,
    "козы": goats,
    "лошади": horses,
    "птица": poultry,
    "кролики": rabbits,
    "пушные звери": fur_animals,
    "пчёлы": bees,
}

# ============================================================
# ТАБЛИЦЫ РИСКОВ
# ============================================================
PEST_DISEASE_LIST = [
    PestDisease("Мучнистая роса пшеницы", "яровая пшеница",
                stage_sensitive=["Выход в трубку", "Колошение"],
                favorable_conditions={"relative_humidity_2m": 80, "temperature_2m": (15, 25)},
                recommendation="Рекомендована профилактическая фунгицидная обработка."),
    PestDisease("Фитофтороз картофеля", "картофель",
                stage_sensitive=["Бутонизация", "Цветение"],
                favorable_conditions={"relative_humidity_2m": 75, "temperature_2m": (10, 25)},
                recommendation="Обработайте медьсодержащими препаратами."),
    PestDisease("Клоп-черепашка", "яровая пшеница",
                stage_sensitive=["Колошение", "Молочная спелость"],
                favorable_conditions={"temperature_2m": (18, 30), "wind_speed_10m": 5},
                recommendation="Провести инсектицидную обработку."),
    PestDisease("Парша яблони", "яблоня",
                stage_sensitive=["Цветение", "Рост плодов"],
                favorable_conditions={"relative_humidity_2m": 80, "precipitation": 2},
                recommendation="Обработка бордоской жидкостью."),
    PestDisease("Малинный жук", "малина",
                stage_sensitive=["Цветение"],
                favorable_conditions={"temperature_2m": (15, 25)},
                recommendation="Опрыскивание инсектицидами до цветения."),
    PestDisease("Крестоцветная блошка", "рапс",
                stage_sensitive=["Всходы", "2-4 листа"],
                favorable_conditions={"temperature_2m": (15, 25), "precipitation": 1},
                recommendation="Обработка пиретроидами при появлении жуков."),
]

CLIMATE_STRESSOR_LIST = [
    ClimateStressor("Тепловой стресс у КРС", "КРС",
                    {"temperature_2m": 28},
                    "Обеспечить вентиляцию, прохладную воду, сократить выпас в полдень."),
    ClimateStressor("Тепловой стресс у свиней", "свиньи",
                    {"temperature_2m": 30},
                    "Увеличить подачу воды, включить системы охлаждения."),
    ClimateStressor("Переохлаждение молодняка овец", "овцы",
                    {"temperature_2m": -5},
                    "Обеспечить обогрев в кошаре, дополнительную подстилку."),
    ClimateStressor("Снижение лёта пчёл", "пчёлы",
                    {"wind_speed_10m": 8, "precipitation": 1},
                    "Лёт ограничен, ульи не беспокоить, проверить корм."),
    ClimateStressor("Тепловой стресс у птицы", "птица",
                    {"temperature_2m": 30},
                    "Усилить вентиляцию, обеспечить охлаждённую воду, снизить плотность посадки."),
    ClimateStressor("Переохлаждение телят", "КРС",
                    {"temperature_2m": -10},
                    "Разместить в обогреваемом профилактории, усилить кормление."),
]

# ============================================================
# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ============================================================
def doy_from_date(d: date) -> int:
    return (d - date(d.year, 1, 1)).days + 1


def find_best_window(operation, forecast_df: pd.DataFrame, today: date) -> Tuple[str, int, List[date]]:
    """
    Ищет в прогнозе на ближайшие 7 дней даты, удовлетворяющие условиям операции.
    Возвращает (статус, рекомендуемый сдвиг, список подходящих дней)
    """
    days = pd.date_range(today, today + timedelta(days=6), freq='D')
    daily = forecast_df.resample('D').agg({
        'temperature_2m': ['min', 'max'],
        'precipitation': 'sum',
        'wind_speed_10m': 'mean',
        'soil_moisture_0_1cm': 'mean',
        'relative_humidity_2m': 'mean'
    })
    daily.columns = ['t_min', 't_max', 'precip_sum', 'wind_avg', 'soil_moist', 'humidity']
    good_days = []
    for day in days:
        if day not in daily.index:
            continue
        row = daily.loc[day]
        if temp_min := getattr(operation, 'temp_min', None):
            if row['t_min'] < temp_min: continue
        if temp_max := getattr(operation, 'temp_max', None):
            if row['t_max'] > temp_max: continue
        if precip_max := getattr(operation, 'precip_max', None):
            if row['precip_sum'] > precip_max: continue
        if wind_max := getattr(operation, 'wind_max', None):
            if row['wind_avg'] > wind_max: continue
        if soil_moist_min := getattr(operation, 'soil_moist_min', None):
            if row['soil_moist'] < soil_moist_min: continue
        if humidity_min := getattr(operation, 'humidity_min', None):
            if row['humidity'] < humidity_min: continue
        if humidity_max := getattr(operation, 'humidity_max', None):
            if row['humidity'] > humidity_max: continue
        good_days.append(day.date())
    if len(good_days) >= 2:
        shift = (good_days[0] - today).days
        return ("✅ Благоприятно", max(0, shift), good_days)
    elif len(good_days) == 1:
        shift = (good_days[0] - today).days
        return ("⚠️ Ограниченно благоприятно", max(0, shift), good_days)
    else:
        return ("❌ Неблагоприятно", 7, [])


def generate_full_report(forecast_df: pd.DataFrame, target_date: date,
                         selected_crops=None, selected_livestock=None) -> Dict:
    if selected_crops is None:
        selected_crops = [wheat_spring, potato, sunflower, cabbage, apple]
    if selected_livestock is None:
        selected_livestock = [cattle, pigs, sheep, poultry, bees]

    ops_recommendations = []
    for crop in selected_crops:
        for stage in crop.stages:
            for op in stage.operations:
                start = op.window_start - 5
                end = op.window_end + 5
                current_doy = doy_from_date(target_date)
                if not (start <= current_doy <= end):
                    continue
                status, shift, good_days = find_best_window(op, forecast_df, target_date)
                ops_recommendations.append({
                    'культура': crop.name_rus,
                    'фаза': stage.name,
                    'операция': op.name,
                    'статус': status,
                    'сдвиг_дней': shift,
                    'подходящие_дни': good_days[:3],
                    'описание': op.description
                })
    for species in selected_livestock:
        for breed in species.breeds:
            for state in breed.states:
                for proc in state.procedures:
                    start = proc.window_start - 5
                    end = proc.window_end + 5
                    current_doy = doy_from_date(target_date)
                    if not (start <= current_doy <= end):
                        continue
                    status, shift, good_days = find_best_window(proc, forecast_df, target_date)
                    ops_recommendations.append({
                        'вид': species.name_rus,
                        'порода': breed.name_rus,
                        'состояние': state.name,
                        'операция': proc.name,
                        'статус': status,
                        'сдвиг_дней': shift,
                        'подходящие_дни': good_days[:3],
                        'описание': proc.description
                    })

    risk_alerts = []
    for pest in PEST_DISEASE_LIST:
        short_forecast = forecast_df.iloc[:72]
        cond_met = True
        if 'relative_humidity_2m' in pest.favorable_conditions:
            if short_forecast['relative_humidity_2m'].mean() < pest.favorable_conditions['relative_humidity_2m']:
                cond_met = False
        if 'temperature_2m' in pest.favorable_conditions:
            t_range = pest.favorable_conditions['temperature_2m']
            avg_temp = short_forecast['temperature_2m'].mean()
            if not (t_range[0] <= avg_temp <= t_range[1]):
                cond_met = False
        if 'precipitation' in pest.favorable_conditions:
            if short_forecast['precipitation'].sum() < pest.favorable_conditions['precipitation']:
                cond_met = False
        if cond_met:
            risk_alerts.append({
                'название': pest.name,
                'культура': pest.culture,
                'рекомендация': pest.recommendation
            })

    stress_alerts = []
    for stressor in CLIMATE_STRESSOR_LIST:
        daily_max = forecast_df.resample('D')['temperature_2m'].max()
        for day, val in daily_max.items():
            if 'temperature_2m' in stressor.threshold_params:
                if val >= stressor.threshold_params['temperature_2m']:
                    stress_alerts.append({
                        'стрессор': stressor.name,
                        'вид': stressor.species,
                        'дата': day.strftime('%Y-%m-%d'),
                        'меры': stressor.mitigation
                    })
        if 'wind_speed_10m' in stressor.threshold_params:
            daily_wind = forecast_df.resample('D')['wind_speed_10m'].mean()
            for day, val in daily_wind.items():
                if val >= stressor.threshold_params['wind_speed_10m']:
                    stress_alerts.append({
                        'стрессор': stressor.name,
                        'вид': stressor.species,
                        'дата': day.strftime('%Y-%m-%d'),
                        'меры': stressor.mitigation
                    })

    summary = forecast_df.resample('D').agg({
        'temperature_2m': ['min', 'max'],
        'precipitation': 'sum',
        'wind_speed_10m': 'mean'
    })

    return {
        'operations': ops_recommendations,
        'risk_alerts': risk_alerts,
        'stress_alerts': stress_alerts,
        'meteo_summary': summary
    }


# ============================================================
# АРХИВ
# ============================================================
ARCHIVE_DIR = 'archive'

def ensure_archive_dir():
    os.makedirs(ARCHIVE_DIR, exist_ok=True)

def archive_forecast(df: pd.DataFrame):
    ensure_archive_dir()
    for date, group in df.groupby(df.index.date):
        filename = f"weather_{date}.csv"
        filepath = os.path.join(ARCHIVE_DIR, filename)
        group.to_csv(filepath, index=True, encoding='utf-8')
        print(f"  [архив] обновлён {filepath}")


# ============================================================
# ПОГОДА
# ============================================================
def get_weather_forecast_7days(lat, lon):
    params = [
        'temperature_2m',
        'relative_humidity_2m',
        'precipitation',
        'soil_temperature_0cm',
        'soil_temperature_6cm',
        'soil_moisture_0_1cm',
        'soil_moisture_1_3cm',
        'wind_speed_10m',
        'wind_gusts_10m',
        'cloud_cover',
        'pressure_msl',
        'et0_fao_evapotranspiration',
        'precipitation_probability',
    ]
    url = "https://api.open-meteo.com/v1/forecast"
    request_params = {
        'latitude': lat,
        'longitude': lon,
        'hourly': params,
        'forecast_days': 7,
        'timezone': 'auto'
    }
    response = requests.get(url, params=request_params)
    data = response.json()
    hourly = data['hourly']
    df = pd.DataFrame(hourly)
    df['datetime'] = pd.to_datetime(df['time'])
    df.set_index('datetime', inplace=True)
    df.sort_index(inplace=True)
    return df


# ============================================================
# MAIN
# ============================================================
LAT, LON = 53.347915, 83.776510  # Барнаул

def main():
    print("🌾 Агрометеорологический советчик (расширенный)")
    print("=" * 55)

    print("\n📡 Получение прогноза...")
    df = get_weather_forecast_7days(LAT, LON)
    print(f"Прогноз с {df.index.min()} по {df.index.max()}")

    print("\n💾 Архивация...")
    archive_forecast(df)

    today = date.today()
    print(f"\n📅 Оценка календаря на {today} и последующие 6 дней:")

    report = generate_full_report(df, today)

    print("\n📋 ПЛАНОВЫЕ ОПЕРАЦИИ И СДВИГИ")
    if not report['operations']:
        print("Нет активных операций в ближайшем окне.")
    for rec in report['operations']:
        icon = rec['статус'][0] if rec['статус'] else "?"
        print(f"{icon} {rec.get('культура', rec.get('вид'))} | {rec['операция']} — {rec['статус']}")
        if rec['сдвиг_дней'] > 0:
            print(f"   Рекомендуемый сдвиг: +{rec['сдвиг_дней']} дн. (благоприятные дни: {rec['подходящие_дни']})")
        else:
            print(f"   Благоприятные дни: {rec['подходящие_дни']}")
        if rec['описание']:
            print(f"   {rec['описание']}")

    print("\n⚠️ ПРЕДУПРЕЖДЕНИЯ О БОЛЕЗНЯХ/ВРЕДИТЕЛЯХ")
    if not report['risk_alerts']:
        print("Нет активных предупреждений.")
    for alert in report['risk_alerts']:
        print(f"🦠 {alert['название']} (культура: {alert['культура']})")
        print(f"   Рекомендация: {alert['рекомендация']}")

    print("\n🌡️ КЛИМАТИЧЕСКИЕ СТРЕССЫ ДЛЯ ЖИВОТНЫХ")
    if not report['stress_alerts']:
        print("Нет опасных стрессов.")
    for s in report['stress_alerts']:
        print(f"{s['стрессор']} ({s['вид']}) на {s['дата']}: {s['меры']}")

    print("\n☀️ МЕТЕОСВОДКА НА НЕДЕЛЮ (день/макс.t/мин.t/осадки/ветер)")
    summary = report['meteo_summary']
    for day, row in summary.iterrows():
        print(f"  {day.strftime('%d.%m')}: t {row[('temperature_2m','min')]:.0f}…{row[('temperature_2m','max')]:.0f}°C, "
              f"осадки {row[('precipitation','sum')]:.1f} мм, ветер {row[('wind_speed_10m','mean')]:.1f} м/с")

    print("\n" + "=" * 55)
    print("✅ Анализ завершён. Данные архивированы.")


if __name__ == "__main__":
    main()

🌾 Агрометеорологический советчик (расширенный)

📡 Получение прогноза...
Прогноз с 2026-05-06 00:00:00 по 2026-05-12 23:00:00

💾 Архивация...
  [архив] обновлён archive\weather_2026-05-06.csv
  [архив] обновлён archive\weather_2026-05-07.csv
  [архив] обновлён archive\weather_2026-05-08.csv
  [архив] обновлён archive\weather_2026-05-09.csv
  [архив] обновлён archive\weather_2026-05-10.csv
  [архив] обновлён archive\weather_2026-05-11.csv
  [архив] обновлён archive\weather_2026-05-12.csv

📅 Оценка календаря на 2026-05-06 и последующие 6 дней:

📋 ПЛАНОВЫЕ ОПЕРАЦИИ И СДВИГИ
✅ Яровая пшеница (мягкая) | Закрытие влаги — ✅ Благоприятно
   Благоприятные дни: [datetime.date(2026, 5, 6), datetime.date(2026, 5, 7), datetime.date(2026, 5, 8)]
❌ Яровая пшеница (мягкая) | Посев — ❌ Неблагоприятно
   Рекомендуемый сдвиг: +7 дн. (благоприятные дни: [])
   Прогрев почвы до +8…+10 °C.
❌ Подсолнечник | Посев — ❌ Неблагоприятно
   Рекомендуемый сдвиг: +7 дн. (благоприятные дни: [])
❌ Яблоня (полукультурка